**사내 공지 CSV를 벡터 스토어에 넣고, 자연어 질문에서 부서 · 연도 조건을 뽑아 메타데이터 필터로 검색하는 검색기를 만들어라.**
* 데이터: id, year, department, title, body 열을 가진 공지 8건을 CSV로 만든다(부서 3개: 인사 · IT · 총무 / 연도 2025 · 2026).
```
id,year,department,title,body
N01,2025,인사,2025년 하반기 승진 심사 일정,승진 심사 서류는 11월 7일까지 인사 시스템에 제출한다.
N02,2026,인사,2026년 연차 촉진 안내,미사용 연차는 10월 31일까지 사용 계획을 등록해야 하며 미등록 시 소멸될 수 있다.
N03,2026,인사,육아휴직 급여 추가 지원 확대,2026년부터 회사 추가 지원 기간이 3개월에서 6개월로 늘어난다.
N04,2025,IT,메일 시스템 점검,12월 6일 02시~06시 메일 시스템 점검으로 메일 송수신이 중단된다.
N05,2026,IT,VPN 클라이언트 교체,10월 20일부터 신규 VPN 클라이언트만 접속할 수 있으니 사전에 설치한다.
N06,2026,IT,사내 AI 포털 오픈,업무용 생성형 AI는 사내 AI 포털에서만 사용하며 로그인은 SSO로 한다.
N07,2025,총무,주차장 도색 공사,11월 15일 지하 2층 주차장이 도색 공사로 폐쇄된다.
N08,2026,총무,사내 식당 운영 시간 변경,10월부터 점심 운영 시간이 11시 30분~13시 30분으로 변경된다.
```
* CSVLoader로 로드한다.

    * content_columns=["title", "body"]: 임베딩할 본문

    * metadata_columns=["id", "year", "department"]: 필터와 출처에 쓸 값

* InMemoryVectorStore에 저장한다.

* SearchPlan Pydantic 모델(query, department, year)과 with_structured_output으로 질문에서 검색 문장과 필터를 추출한다.

    * department, year는 Optional[Literal[...]]로 허용값을 제한하고, 질문에 없으면 None

    * "올해"는 2026년으로 해석하도록 system 메시지에 명시

* 추출한 필터로 similarity_search(..., filter=함수)를 실행한다. 필터 결과가 없으면 필터 없이 다시 검색한다.

* 테스트 질문 4개(부서 지정, "올해" + 부서, 연도만 지정, 조건 없음)마다 추출된 검색어 · 필터와 검색 결과(ID · 연도 · 부서 · 제목)를 출력한다.
```
[
    "IT 부서 공지 중에 접속 관련 내용 있어?",
    "올해 인사팀 휴직 관련 공지 알려줘",
    "2025년 공지 중 공사나 점검 일정",
    "점심 시간 바뀌었어?",
]
```

In [13]:
import csv
import warnings
from pathlib import Path
from typing import Literal, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_community.document_loaders import CSVLoader
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

warnings.filterwarnings("ignore", category=DeprecationWarning)
load_dotenv()

chat_model = ChatOpenAI(model="gpt-6-luna", temperature=0, reasoning={"effort": "none"})
embedder = OpenAIEmbeddings(model="text-embedding-3-small")

In [14]:
# 공지 8건을 csv 모듈로 저장
NOTICE_CSV = Path("notices.csv")
NOTICE_ROWS = [
    ("id", "year", "department", "title", "body"),
    ("N01", 2025, "인사", "2025년 하반기 승진 심사 일정", "승진 심사 서류는 11월 7일까지 인사 시스템에 제출한다."),
    ("N02", 2026, "인사", "2026년 연차 촉진 안내", "미사용 연차는 10월 31일까지 사용 계획을 등록해야 하며 미등록 시 소멸될 수 있다."),
    ("N03", 2026, "인사", "육아휴직 급여 추가 지원 확대", "2026년부터 회사 추가 지원 기간이 3개월에서 6개월로 늘어난다."),
    ("N04", 2025, "IT", "메일 시스템 점검", "12월 6일 02시~06시 메일 시스템 점검으로 메일 송수신이 중단된다."),
    ("N05", 2026, "IT", "VPN 클라이언트 교체", "10월 20일부터 신규 VPN 클라이언트만 접속할 수 있으니 사전에 설치한다."),
    ("N06", 2026, "IT", "사내 AI 포털 오픈", "업무용 생성형 AI는 사내 AI 포털에서만 사용하며 로그인은 SSO로 한다."),
    ("N07", 2025, "총무", "주차장 도색 공사", "11월 15일 지하 2층 주차장이 도색 공사로 폐쇄된다."),
    ("N08", 2026, "총무", "사내 식당 운영 시간 변경", "10월부터 점심 운영 시간이 11시 30분~13시 30분으로 변경된다."),
]
with NOTICE_CSV.open("w", encoding="utf-8", newline="") as fp:
    csv.writer(fp).writerows(NOTICE_ROWS)

In [15]:
# 본문 열은 임베딩 대상, 나머지 열은 metadata로 분리해 벡터 스토어에 저장
notice_docs = CSVLoader(
    file_path=str(NOTICE_CSV),
    encoding="utf-8",
    content_columns=["title", "body"],
    metadata_columns=["id", "year", "department"],
).load()

notice_store = InMemoryVectorStore.from_documents(notice_docs, embedding=embedder)

print(f"적재 문서 수: {len(notice_docs)}")
print("첫 문서 metadata:", notice_docs[0].metadata)

적재 문서 수: 8
첫 문서 metadata: {'source': 'notices.csv', 'row': 0, 'id': 'N01', 'year': '2025', 'department': '인사'}


In [16]:
# 질문 분석 스키마: 검색 문장 + 선택적 필터 (허용값 밖의 값은 스키마 단계에서 차단)
class SearchPlan(BaseModel):
    query: str = Field(description="부서 · 연도 표현을 제외한 핵심 검색 문장")
    department: Optional[Literal["인사", "IT", "총무"]] = Field(None, description="질문에 부서가 드러날 때만 지정")
    year: Optional[Literal[2025, 2026]] = Field(None, description="질문에 연도가 드러날 때만 지정")

PLANNER_RULE = (
    "사내 공지 검색 질문에서 검색 문장, 부서, 연도를 분리합니다. "
    "부서나 연도가 질문에 드러나지 않으면 null로 둡니다. "
    "'올해'는 2026년으로 해석합니다."
)
planner = chat_model.with_structured_output(SearchPlan)

def analyze(question: str) -> SearchPlan:
    return planner.invoke([SystemMessage(PLANNER_RULE), HumanMessage(question)])

In [17]:
# 필터 조건 dict와 이를 검사하는 함수를 함께 생성 (CSVLoader metadata 값은 문자열)
def build_filter(plan: SearchPlan):
    wanted = {}
    if plan.department:
        wanted["department"] = plan.department
    if plan.year:
        wanted["year"] = str(plan.year)

    def accept(doc) -> bool:
        return all(doc.metadata.get(name) == value for name, value in wanted.items())

    return wanted, accept

def find_notices(question: str, k: int = 3):
    plan = analyze(question)
    text = plan.query or question
    wanted, accept = build_filter(plan)

    hits = notice_store.similarity_search(text, k=k, filter=accept) if wanted else []
    relaxed = False
    if not hits:  # 필터가 없었거나 필터 결과가 비었으면 전체 대상으로 검색
        hits = notice_store.similarity_search(text, k=k)
        relaxed = bool(wanted)
    return text, wanted, relaxed, hits

In [18]:
TEST_QUESTIONS = [
    "IT 부서 공지 중에 접속 관련 내용 있어?",
    "올해 인사팀 휴직 관련 공지 알려줘",
    "2025년 공지 중 공사나 점검 일정",
    "점심 시간 바뀌었어?",
]

for number, question in enumerate(TEST_QUESTIONS, start=1):
    text, wanted, relaxed, hits = find_notices(question)
    print(f"[{number}] {question}")
    print(f"    검색어: {text}")
    print(f"    필터: {wanted or '없음'}" + (" -> 결과 없음, 필터 해제 후 재검색" if relaxed else ""))
    for doc in hits:
        meta = doc.metadata
        title = doc.page_content.split("\n")[0].removeprefix("title: ")
        print(f"      {meta['id']} | {meta['year']} | {meta['department']} | {title}")
    print()

[1] IT 부서 공지 중에 접속 관련 내용 있어?
    검색어: 접속 관련 내용
    필터: {'department': 'IT'}
      N05 | 2026 | IT | VPN 클라이언트 교체
      N04 | 2025 | IT | 메일 시스템 점검
      N06 | 2026 | IT | 사내 AI 포털 오픈

[2] 올해 인사팀 휴직 관련 공지 알려줘
    검색어: 휴직 관련 공지
    필터: {'department': '인사', 'year': '2026'}
      N03 | 2026 | 인사 | 육아휴직 급여 추가 지원 확대
      N02 | 2026 | 인사 | 2026년 연차 촉진 안내

[3] 2025년 공지 중 공사나 점검 일정
    검색어: 공사나 점검 일정
    필터: {'year': '2025'}
      N04 | 2025 | IT | 메일 시스템 점검
      N01 | 2025 | 인사 | 2025년 하반기 승진 심사 일정
      N07 | 2025 | 총무 | 주차장 도색 공사

[4] 점심 시간 바뀌었어?
    검색어: 점심 시간 변경
    필터: 없음
      N08 | 2026 | 총무 | 사내 식당 운영 시간 변경
      N04 | 2025 | IT | 메일 시스템 점검
      N05 | 2026 | IT | VPN 클라이언트 교체



---

**마크다운 제품 매뉴얼 2개로 RAG 체인을 만들고, 답변 · 찾음 여부 · 참고 섹션을 JSON으로 반환하라.**

* 데이터: 로봇청소기 · 에어프라이어 매뉴얼을 #, ##, ### 헤더 구조의 마크다운으로 준비한다.
```
-- robot_cleaner.md

# 로봇청소기 RC-9 사용 설명서
## 설치
### 충전 스테이션 위치
충전 스테이션은 벽에 붙이고 좌우 0.5m, 앞쪽 1.5m 이내에 장애물이 없는 곳에 둔다.
### 첫 충전
처음 사용할 때는 4시간 이상 완전히 충전한 뒤 청소를 시작한다.
## 청소 모드
### 자동 모드
전체 공간을 지도로 만든 뒤 구역 순서대로 청소한다. 한 번 충전으로 최대 150분 동작한다.
### 집중 모드
지정한 1m x 1m 구역을 두 번 반복해 청소한다.
## 문제 해결
### 충전이 되지 않을 때
충전 단자를 마른 천으로 닦고, 스테이션 전원 어댑터가 연결되어 있는지 확인한다.
### 오류 코드 E3
바퀴에 머리카락이나 이물질이 감긴 상태다. 전원을 끄고 바퀴를 분리해 청소한다.
```
```
-- air_fryer.md

# 에어프라이어 AF-5 사용 설명서
## 사용 방법
### 예열
조리 전 180도에서 3분간 예열하면 조리 시간이 짧아지고 결과가 고르게 나온다.
### 조리 용량
바스켓에는 최대 1.2kg까지 넣을 수 있으며, 재료가 겹치지 않게 펼쳐 담는다.
## 관리
### 세척
바스켓과 트레이는 식기세척기 사용이 가능하다. 본체는 전원을 뽑고 젖은 천으로 닦는다.
## 문제 해결
### 흰 연기가 날 때
기름이 많은 재료에서 기름이 떨어져 타는 경우다. 트레이의 기름을 버리고 다시 조리한다.
```
* 2단계로 분할한다.

    * MarkdownHeaderTextSplitter로 헤더 단위 분할(헤더 → metadata h1/h2/h3)

    * RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)로 긴 섹션을 한 번 더 분할

    * metadata에 file과 section("문제 해결 > 오류 코드 E3" 형식)을 추가

* FAISS에 저장하고 MMR 검색기(k=3, fetch_k=8)를 만든다.

* 출력 스키마 ManualAnswer

    * found: bool, answer: str, sources: list[Source(file, section)]

    * 체인은 RunnableParallel(docs=retriever, question=...)로 검색 결과를 보존하고, with_structured_output(ManualAnswer)로 답변을 만든다.

* 후처리 함수(RunnableLambda)

    * sources 중 실제 검색되지 않은 출처는 제거

    * found=false면 고객센터 안내 문구로 교체

* 테스트 질문 4개(매뉴얼에 있는 질문 3개 + 없는 질문 1개)의 결과를 JSON으로 출력한다.
```
["로봇청소기에 E3가 떠요", "에어프라이어 바스켓 식기세척기 돼?",
 "처음 쓸 때 충전은 얼마나 해야 해?", "에어프라이어 보증 기간은?"]
```

In [19]:
import json
import warnings
from operator import itemgetter
from pathlib import Path

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore", category=DeprecationWarning)
load_dotenv()

chat = ChatOpenAI(model="gpt-6-luna", temperature=0, reasoning={"effort": "none"})
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [20]:
# 매뉴얼 2개를 마크다운 파일로 저장
MANUAL_DIR = Path("manuals")
MANUAL_DIR.mkdir(exist_ok=True)

MANUAL_TEXTS = {
    "robot_cleaner.md": """# 로봇청소기 RC-9 사용 설명서
## 설치
### 충전 스테이션 위치
충전 스테이션은 벽에 붙이고 좌우 0.5m, 앞쪽 1.5m 이내에 장애물이 없는 곳에 둔다.
### 첫 충전
처음 사용할 때는 4시간 이상 완전히 충전한 뒤 청소를 시작한다.
## 청소 모드
### 자동 모드
전체 공간을 지도로 만든 뒤 구역 순서대로 청소한다. 한 번 충전으로 최대 150분 동작한다.
### 집중 모드
지정한 1m x 1m 구역을 두 번 반복해 청소한다.
## 문제 해결
### 충전이 되지 않을 때
충전 단자를 마른 천으로 닦고, 스테이션 전원 어댑터가 연결되어 있는지 확인한다.
### 오류 코드 E3
바퀴에 머리카락이나 이물질이 감긴 상태다. 전원을 끄고 바퀴를 분리해 청소한다.
""",
    "air_fryer.md": """# 에어프라이어 AF-5 사용 설명서
## 사용 방법
### 예열
조리 전 180도에서 3분간 예열하면 조리 시간이 짧아지고 결과가 고르게 나온다.
### 조리 용량
바스켓에는 최대 1.2kg까지 넣을 수 있으며, 재료가 겹치지 않게 펼쳐 담는다.
## 관리
### 세척
바스켓과 트레이는 식기세척기 사용이 가능하다. 본체는 전원을 뽑고 젖은 천으로 닦는다.
## 문제 해결
### 흰 연기가 날 때
기름이 많은 재료에서 기름이 떨어져 타는 경우다. 트레이의 기름을 버리고 다시 조리한다.
""",
}
for name, body in MANUAL_TEXTS.items():
    (MANUAL_DIR / name).write_text(body, encoding="utf-8")

In [21]:
# 1단계: 헤더 기준 분할, 2단계: 길이 기준 분할
by_header = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")])
by_length = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)

def split_manual(path: Path):
    pieces = by_length.split_documents(by_header.split_text(path.read_text(encoding="utf-8")))
    for piece in pieces:
        heads = [piece.metadata.get(key) for key in ("h2", "h3")]
        piece.metadata["file"] = path.name
        piece.metadata["section"] = " > ".join(h for h in heads if h)
    return pieces

manual_chunks = [chunk for path in sorted(MANUAL_DIR.glob("*.md")) for chunk in split_manual(path)]
print(f"청크 수: {len(manual_chunks)}")
for chunk in manual_chunks[:3]:
    print(f"  {chunk.metadata['file']} | {chunk.metadata['section']} | {chunk.page_content[:25]}")

manual_index = FAISS.from_documents(manual_chunks, embeddings)
manual_retriever = manual_index.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 8})

청크 수: 10
  air_fryer.md | 사용 방법 > 예열 | 조리 전 180도에서 3분간 예열하면 조리 시
  air_fryer.md | 사용 방법 > 조리 용량 | 바스켓에는 최대 1.2kg까지 넣을 수 있으며
  air_fryer.md | 관리 > 세척 | 바스켓과 트레이는 식기세척기 사용이 가능하다.


In [22]:
# 출력 스키마와 답변 생성 프롬프트
class Source(BaseModel):
    file: str = Field(description="근거로 쓴 파일명")
    section: str = Field(description="근거로 쓴 섹션 경로, 예: 문제 해결 > 오류 코드 E3")

class ManualAnswer(BaseModel):
    found: bool = Field(description="제공된 문서에서 답을 찾았는지 여부")
    answer: str = Field(description="문서에 근거한 답변, 못 찾았으면 빈 문자열")
    sources: list[Source] = Field(default_factory=list, description="답변에 실제로 쓴 섹션 목록")

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "제품 매뉴얼 안내 담당자로서 <manual> 안의 내용만으로 답변합니다.\n"
     "내용에 답이 없으면 found를 false로 두고 추측하지 않습니다.\n"
     "sources에는 답변에 직접 사용한 파일과 섹션만 적습니다.\n\n"
     "<manual>\n{manual}\n</manual>"),
    ("human", "{question}"),
])
answer_llm = chat.with_structured_output(ManualAnswer)

In [23]:
NOT_FOUND_MESSAGE = "매뉴얼에서 확인되지 않는 내용입니다. 고객센터(1588-0000)로 문의해 주세요."

def render_manual(docs) -> str:
    return "\n".join(
        f"[{d.metadata['file']} / {d.metadata['section']}] {d.page_content}" for d in docs
    )

def postprocess(state: dict) -> dict:
    result: ManualAnswer = state["result"]
    if not result.found:
        return {"found": False, "answer": NOT_FOUND_MESSAGE, "sources": []}
    # 실제 검색된 섹션과 일치하는 출처만 유지
    retrieved = {(d.metadata["file"], d.metadata["section"]) for d in state["docs"]}
    kept = [s.model_dump() for s in result.sources if (s.file, s.section) in retrieved]
    return {"found": True, "answer": result.answer, "sources": kept}

generate = (
    RunnableLambda(lambda x: {"manual": render_manual(x["docs"]), "question": x["question"]})
    | answer_prompt
    | answer_llm
)

manual_chain = (
    RunnableParallel(docs=manual_retriever, question=RunnablePassthrough())
    | RunnableParallel(docs=itemgetter("docs"), result=generate)
    | RunnableLambda(postprocess)
)

In [24]:
MANUAL_QUESTIONS = [
    "로봇청소기에 E3가 떠요",
    "에어프라이어 바스켓 식기세척기 돼?",
    "처음 쓸 때 충전은 얼마나 해야 해?",
    "에어프라이어 보증 기간은?",
]

outputs = [{"question": q, **manual_chain.invoke(q)} for q in MANUAL_QUESTIONS]
print(json.dumps(outputs, ensure_ascii=False, indent=2))

[
  {
    "question": "로봇청소기에 E3가 떠요",
    "found": false,
    "answer": "매뉴얼에서 확인되지 않는 내용입니다. 고객센터(1588-0000)로 문의해 주세요.",
    "sources": []
  },
  {
    "question": "에어프라이어 바스켓 식기세척기 돼?",
    "found": true,
    "answer": "네. 에어프라이어 바스켓은 식기세척기 사용이 가능합니다.",
    "sources": [
      {
        "file": "air_fryer.md",
        "section": "관리 > 세척"
      }
    ]
  },
  {
    "question": "처음 쓸 때 충전은 얼마나 해야 해?",
    "found": true,
    "answer": "처음 사용할 때는 4시간 이상 완전히 충전한 뒤 청소를 시작하세요.",
    "sources": [
      {
        "file": "robot_cleaner.md",
        "section": "설치 > 첫 충전"
      }
    ]
  },
  {
    "question": "에어프라이어 보증 기간은?",
    "found": false,
    "answer": "매뉴얼에서 확인되지 않는 내용입니다. 고객센터(1588-0000)로 문의해 주세요.",
    "sources": []
  }
]
